# SegmentAE Example: Grid Search Optimization

This notebook demonstrates how to use SegmentAE_Optimizer for automated grid search to find the optimal configuration across multiple autoencoders, clustering methods, and threshold ratios.

## Features demonstrated:
- EnsembleAutoencoder with multiple architectures
- Custom basic autoencoder implementation
- Automated grid search optimization
- Performance leaderboard and best configuration selection

## Import Dependencies

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential

from segmentae.autoencoders import EnsembleAutoencoder
from segmentae.data_sources import load_dataset
from segmentae.metrics import metrics_classification
from segmentae.optimization import SegmentAE_Optimizer
from segmentae.processing.preprocessing import Preprocessing

## Data Loading

The labeled data is split in two: a **validation** set, which the optimizer uses to select the best configuration, and a **hold-out** set that the search never sees, used at the end to report an unbiased final performance.

In [ ]:
train, test, target = load_dataset(
    dataset_selection="htru2_dataset",  # Options: 'default_credit_card', 'htru2_dataset', 'shuttle_148'
    split_ratio=0.75,
)

# Validation set: selects the configuration || Hold-out set: final, unbiased evaluation
valid, holdout = train_test_split(test, train_size=0.6, random_state=5, stratify=test[target])

X_train = train.drop(columns=[target]).copy()
X_valid, y_valid = valid.drop(columns=[target]).copy(), valid[target].astype(int)
X_holdout, y_holdout = holdout.drop(columns=[target]).copy(), holdout[target].astype(int)

print("Dataset: HTRU2 Pulsar Detection")
print(f"  Training samples: {len(X_train)}")
print(f"  Validation samples: {len(X_valid)}")
print(f"  Hold-out samples: {len(X_holdout)}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Target classes: {sorted(y_valid.unique())}")

## Data Preprocessing

In [ ]:
# MinMaxScaler maps the features to [0, 1], the range a sigmoid output layer can
# reconstruct. With "StandardScaler" or "RobustScaler", use a linear output layer instead.
pr = Preprocessing(
    encoder="IFrequencyEncoder",  # "LabelEncoder", "OneHotEncoder"
    scaler="MinMaxScaler",  # "StandardScaler", "RobustScaler"
    imputer=None,
)

pr.fit(X=X_train)
X_train = pr.transform(X=X_train)
X_valid = pr.transform(X=X_valid)
X_holdout = pr.transform(X=X_holdout)

print("✓ Preprocessing completed")
print(f"  Processed shape: {X_train.shape}")

## Check GPU Availability

In [ ]:
n_gpus = len(tf.config.list_physical_devices("GPU"))
print("Num GPUs Available: ", n_gpus)
if n_gpus > 0:
    print("✓ GPU acceleration available")
else:
    print("⚠️ Running on CPU only")

## Ensemble Autoencoder Architecture

Build an ensemble autoencoder that combines multiple architectures for robustness.

In [ ]:
# EnsembleAutoencoder combines multiple architectures for robustness
print("Building Ensemble Autoencoder...")
ensembleAutoencoder = EnsembleAutoencoder(
    n_autoencoders=3,
    hidden_dims=[[24, 12, 8, 4], [64, 32, 16, 4], [20, 10, 5]],
    encoder_activations=["relu", "tanh", "relu"],
    decoder_activations=["relu", "tanh", "relu"],
    optimizers=["adam", "sgd", "rmsprop"],
    learning_rates=[0.001, 0.01, 0.005],
    epochs_list=[10, 15, 20],
    val_size_list=[0.15, 0.15, 0.15],
    stopping_patients=[30, 30, 15],
    dropout_rates=[0.1, 0.2, 0.3],
    batch_sizes=[None, None, 128],
    use_batch_norm=[False, False, False],
    random_state=42,
)

print("  Configuration:")
print(f"    - Number of autoencoders: {ensembleAutoencoder.n_autoencoders}")
print(f"    - Architectures: {ensembleAutoencoder.hidden_dims}")
print(f"    - Optimizers: {ensembleAutoencoder.optimizers}")

In [ ]:
print("Training ensemble autoencoder...")
ensembleAutoencoder.fit(X_train)
print("✓ Ensemble autoencoder trained successfully!")

## Custom Basic Autoencoder

Create a simple custom autoencoder for comparison.

In [ ]:
print("Building Custom Basic Autoencoder...")

encoder = Sequential(
    [
        Input(shape=(X_train.shape[1],)),
        Dense(32, activation="relu"),
        Dense(24, activation="relu"),
        Dense(16, activation="relu"),
        Dense(8, activation="relu"),
    ]
)

decoder = Sequential(
    [
        Input(shape=(8,)),
        Dense(16, activation="relu"),
        Dense(24, activation="relu"),
        Dense(32, activation="relu"),
        Dense(X_train.shape[1], activation="sigmoid"),  # reconstructs MinMax-scaled values
    ]
)

basicAutoencoder = Sequential([encoder, decoder])
basicAutoencoder.compile(optimizer="adam", loss="mse")

print("  Architecture: Input → 32 → 24 → 16 → 8 → 16 → 24 → 32 → Output")
print(f"  Total parameters: {basicAutoencoder.count_params():,}")

In [ ]:
print("Training basic autoencoder...")
basic_history = basicAutoencoder.fit(X_train, X_train, epochs=50, batch_size=None, validation_split=0.1, verbose=1)
print("✓ Basic autoencoder trained successfully!")

## Grid Search Optimization Setup

Configure the SegmentAE_Optimizer to test multiple configurations automatically.

In [ ]:
# Using string-based configuration (backward compatible)
print("Setting up SegmentAE Optimizer...")

optimizer = SegmentAE_Optimizer(
    autoencoder_models=[ensembleAutoencoder, basicAutoencoder],
    n_clusters_list=[1, 2, 3],
    cluster_models=["KMeans", "MiniBatchKMeans", "GMM"],
    threshold_ratios=[1, 1.5, 2, 4],
    performance_metric="Accuracy",
    verbose=True,  # Set to False to silence the progress output
)

print("  Optimization configuration:")
print("    - Autoencoders: 2 models (Ensemble + Basic)")
print(f"    - Cluster numbers: {optimizer.n_clusters_list}")
print(f"    - Clustering methods: {optimizer.cluster_models}")
print(f"    - Threshold ratios: {optimizer.threshold_ratios}")
print(f"    - Optimization metric: {optimizer.performance_metric}")

total_configs = len(optimizer.autoencoder_models) * len(optimizer.n_clusters_list) * len(optimizer.cluster_models) * len(optimizer.threshold_ratios)
print(f"    - Total configurations to test: {total_configs}")

## Run Grid Search Optimization

Execute the automated optimization process.

In [ ]:
# Run optimization: configurations are scored on the validation set
print("\n Running grid search optimization...")
print("This may take a few minutes...\n")

sg = optimizer.optimize(X_train, X_valid, y_valid)

print("\n✓ Optimization completed!")

## Optimization Results Analysis

In [ ]:
preds_valid = sg.preds_test
recon_metrics_valid = sg.reconstruction_test
leaderboard = optimizer.leaderboard

print("=" * 60)
print("🏆 OPTIMIZATION RESULTS")
print("=" * 60)
print(f"\nBest Validation Performance: {optimizer.best_performance:.4f}")
print("\nBest Configuration:")
print(f"  - Model: {'Ensemble' if optimizer.best_autoencoder is ensembleAutoencoder else 'Basic Custom'}")
print(f"  - Clusters: {optimizer.best_n_clusters}")
print(f"  - Clustering Algorithm: {optimizer.best_cluster_model}")
print(f"  - Threshold Ratio: {optimizer.best_threshold_ratio}")

## Performance Leaderboard

View the top performing configurations.

In [ ]:
print("\nTop 10 Configurations:")
print("=" * 80)

# Autoencoder_Index is the position of the model in `autoencoder_models`
autoencoder_names = {0: 'Ensemble', 1: 'Basic'}

top_10 = leaderboard.head(10).copy()
top_10['Autoencoder_Type'] = top_10['Autoencoder_Index'].map(autoencoder_names)

display_cols = ['Autoencoder_Type', 'Cluster', 'N_Clusters', 'Threshold Ratio', 'Accuracy']
print(top_10[display_cols].to_string(index=False))

## Final Anomaly Detection with Optimal Configuration

The optimal configuration is applied to the hold-out set, which played no part in selecting it.

In [ ]:
threshold = optimizer.best_threshold_ratio

# Hold-out data was not used to select the configuration
predictions = sg.detections(input_data=X_holdout, threshold_ratio=threshold)

final_metrics = metrics_classification(y_holdout, predictions["Predicted Anomalies"])

print("\n" + "=" * 50)
print(" FINAL HOLD-OUT PERFORMANCE")
print("=" * 50)
print("\nUsing optimal configuration:")
print(f"  - Threshold ratio: {threshold}")
print(f"  - Clustering: {optimizer.best_cluster_model} (n={optimizer.best_n_clusters})")
print("\nPerformance metrics:")
for metric in final_metrics.columns:
    print(f"  {metric}: {final_metrics[metric].iloc[0]:.4f}")

# Detection summary
n_anomalies = predictions["Predicted Anomalies"].sum()
anomaly_rate = n_anomalies / len(predictions)
print("\nDetection summary:")
print(f"  Anomalies detected: {n_anomalies:,} / {len(predictions):,}")
print(f"  Anomaly rate: {anomaly_rate:.2%}")

## Results Visualization

In [ ]:
import matplotlib.pyplot as plt

# Performance comparison by autoencoder type and by clustering method
viz_data = leaderboard.copy()
viz_data['Autoencoder_Type'] = viz_data['Autoencoder_Index'].map(autoencoder_names)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

viz_data.boxplot(column='Accuracy', by='Autoencoder_Type', ax=axes[0])
axes[0].set_title('Performance by Autoencoder Type')
axes[0].set_xlabel('')
axes[0].set_ylabel('Accuracy')

viz_data.boxplot(column='Accuracy', by='Cluster', ax=axes[1])
axes[1].set_title('Performance by Clustering Method')
axes[1].set_xlabel('')
axes[1].set_ylabel('Accuracy')

fig.suptitle('')
plt.tight_layout()
plt.show()

print("\nVisualization shows performance distribution across different configurations")